# Task 1: Aspect Term Extraction (ATE)
## Baselines: TextCNN-CRF, RNN-CRF, LSTM-CRF, GRU-CRF, BiLSTM-CRF, BiGRU-CRF

ATE = Sequence Labeling: Gian nhan B-ASPECT / I-ASPECT / O cho tung tu.
Notebook nay dong thoi **save** vocab va embeddings de cac notebook sau tai su dung.


## 1. Setup


In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

from src.utils.preprocess import (
    load_raw_data, segment_items, build_vocab,
    train_w2v_embeddings, save_preprocessed, load_preprocessed
)
from src.ate.ate_dataset import ATEDataset, BIO_TAGS, NUM_TAGS
from src.ate.ate_model import build_ate_model
from src.utils.engine import train_ate_model, predict_ate
from src.utils.metrics import bio_tags_to_spans, evaluate_spans_f1, token_accuracy
from src.utils.visualization import plot_training_curves, plot_model_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42); np.random.seed(42)
print(f"Device: {device}")


Device: cuda


## 2. Load Data, Segment, Build Vocab & Embeddings
Chi can chay 1 lan. Ket qua se duoc SAVE de Notebook 02, 03, 04 dung lai.


In [4]:
DATA_DIR = os.path.join("..", "..", "data")
PREP_DIR = os.path.join("..", "..", "preprocessed")

# Load & segment
train_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "train.jsonl")))
dev_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "dev.jsonl")))
test_items = segment_items(load_raw_data(os.path.join(DATA_DIR, "test.jsonl")))
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")

# Build vocab (them [ASP] luon de Notebook 02 dung duoc)
all_texts = [item["text"] for item in train_items + dev_items + test_items]
word2idx = build_vocab(all_texts, min_freq=2, special_tokens=["[ASP]"])
VOCAB_SIZE = len(word2idx)
print(f"Vocab: {VOCAB_SIZE}")

# Train Word2Vec
EMB_DIM = 150
emb_matrix = train_w2v_embeddings(all_texts, word2idx, emb_dim=EMB_DIM)

# SAVE de cac notebook sau khong can chay lai
save_preprocessed(PREP_DIR, word2idx, emb_matrix, train_items, dev_items, test_items)


Train: 7785 | Dev: 1112 | Test: 2225
Vocab: 6169
Training Word2Vec 150d...
Embedding Matrix ready. Hit: 6166/6169 (100.0%)
Saved preprocessed data to ..\..\preprocessed/
  - word2idx: 6169 tokens
  - emb_matrix: (6169, 150)
  - train/dev/test: 7785/1112/2225


## 3. Hyperparameters & DataLoaders


In [5]:
MAX_LEN = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 7

train_ds = ATEDataset(train_items, word2idx, MAX_LEN)
dev_ds = ATEDataset(dev_items, word2idx, MAX_LEN)
test_ds = ATEDataset(test_items, word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, BATCH_SIZE)
test_loader = DataLoader(test_ds, BATCH_SIZE)
print(f"Datasets - Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")


Datasets - Train: 7785 | Dev: 1112 | Test: 2225


## 4. Train All Baselines


In [6]:
MODEL_TYPES = ["TextCNN", "RNN", "LSTM", "GRU", "BiLSTM", "BiGRU"]

all_models = {}
all_histories = {}

for model_type in MODEL_TYPES:
    model = build_ate_model(
        model_type=model_type, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM,
        hidden_dim=HIDDEN_DIM, num_tags=NUM_TAGS,
        pretrained_emb=emb_matrix, n_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(device)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{model_type}-CRF | Params: {param_count:,}")

    model, history = train_ate_model(
        model, train_loader, dev_loader, device,
        lr=1e-3, epochs=EPOCHS, patience=PATIENCE, model_name=f"{model_type}-CRF"
    )
    all_models[model_type] = model
    all_histories[model_type] = history



TextCNN-CRF | Params: 1,805,982

  Training TextCNN-CRF


  Ep  1/30 | Train Loss: 55.8228 | Dev Loss: 40.6467 | TokAcc: 0.6160 | LR: 0.001000 | 37.1s ***


  Ep  2/30 | Train Loss: 38.9151 | Dev Loss: 34.6847 | TokAcc: 0.6418 | LR: 0.001000 | 37.6s ***


  Ep  3/30 | Train Loss: 33.2133 | Dev Loss: 30.9817 | TokAcc: 0.6436 | LR: 0.001000 | 37.4s ***


  Ep  4/30 | Train Loss: 29.2077 | Dev Loss: 27.9648 | TokAcc: 0.6543 | LR: 0.001000 | 36.8s ***


...

  Ep 17/30 | Train Loss: 13.7024 | Dev Loss: 14.8168 | TokAcc: 0.6649 | LR: 0.001000 | 39.3s ***


  Ep 18/30 | Train Loss: 13.1442 | Dev Loss: 14.3251 | TokAcc: 0.6645 | LR: 0.001000 | 39.3s


  Ep 19/30 | Train Loss: 12.6623 | Dev Loss: 14.2903 | TokAcc: 0.6623 | LR: 0.001000 | 43.9s


  Ep 20/30 | Train Loss: 12.2736 | Dev Loss: 13.8270 | TokAcc: 0.6750 | LR: 0.001000 | 44.7s ***


KeyboardInterrupt: 

## 5. Evaluate on Test Set


In [ ]:
all_results = []
for model_type, model in all_models.items():
    test_res = predict_ate(model, test_loader, device)

    pred_spans = [bio_tags_to_spans(pt, BIO_TAGS, l) for pt, l in zip(test_res["pred_tags"], test_res["lengths"])]
    true_spans = [bio_tags_to_spans(tt, BIO_TAGS, l) for tt, l in zip(test_res["true_tags"], test_res["lengths"])]
    span_f1 = evaluate_spans_f1(pred_spans, true_spans)
    tok_acc = token_accuracy(test_res["pred_tags"], test_res["true_tags"])

    all_results.append({
        "Model": f"{model_type}-CRF",
        "Span_P": round(span_f1["precision"], 4),
        "Span_R": round(span_f1["recall"], 4),
        "Span_F1": round(span_f1["f1"], 4),
        "Token_Acc": round(tok_acc, 4),
    })
    print(f"{model_type}-CRF | Span F1: {span_f1['f1']:.4f} | TokAcc: {tok_acc:.4f}")

results_df = pd.DataFrame(all_results)
display(results_df.sort_values("Span_F1", ascending=False).style.highlight_max(
    subset=["Span_F1"], color="lightgreen"))


## 6. Visualization


In [ ]:
plot_training_curves(all_histories)
plot_model_comparison(results_df, metric_col="Span_F1", title="ATE: Span-level F1 Comparison")


## 7. Save Results & Best Model


In [ ]:
SAVE_DIR = os.path.join("..", "..", "results", "ate")
os.makedirs(SAVE_DIR, exist_ok=True)

results_df.to_csv(os.path.join(SAVE_DIR, "ate_results.csv"), index=False)

best_name = results_df.loc[results_df["Span_F1"].idxmax(), "Model"]
best_key = best_name.replace("-CRF", "")
torch.save(all_models[best_key].state_dict(), os.path.join(SAVE_DIR, f"best_ate_{best_name}.pt"))
print(f"Best ATE: {best_name} (F1={results_df['Span_F1'].max():.4f}) -> Saved!")
